# Quebec City → TTE format

**Source:** `data/trips.rda` in [github.com/melmasri/traveltimeCLT](https://github.com/melmasri/traveltimeCLT)
(the identical `tripset` lives in `melmasri/traveltimeHMM`).
Anonymised *Mon Trajet* smartphone GPS, Quebec City, 28 Apr – 16 May 2014,
collected by Brisk Synergies. Package licence **GPL-3** (no separate data licence).

**4 914 trips / 322 799 link traversals / 13 235 links.** The data is *already
map-matched*: every row is one road link of one trip with its entry time and its
traversal duration — exactly the R1 signal, without us having to match anything.

**The one thing it does not have: geometry.** `linkID` is an anonymised integer
with no coordinates, no OSM id and no shapefile; the network was never released.
So this notebook produces `matched_trips_quebec.csv` with `Coordinates` filled
with `(None, None)` placeholders (one per link traversal, so the list lengths
still line up) and `edge_list_directed_quebec.csv` built from the transitions the
trips themselves reveal. **No `road_network_unique_osmids_quebec.geojson` can be
produced** — see the caveats at the bottom.

## Target format (the "gold" contract)

Taken from the Harbin files in `datasets.zip`, with the Omsk file naming:

| file | contents |
|---|---|
| `matched_trips_<city>.csv` | unnamed index, `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time` |
| `edge_list_directed_<city>.csv` | `osmid_u`, `osmid_v` — directed transitions between road segments |
| `road_network_unique_osmids_<city>.geojson` | one `LineString` per segment, `osmid` / `original_osmid` / `is_duplicate` / `duplicate_index` / `length` / `highway` / ... |

`Coordinates`, `OSMids` and `Timestamps` are Python-literal lists of **equal
length — one entry per GPS fix**: `(lon, lat)` floats, the segment id the fix was
matched to (a string), and the unix timestamp in seconds.
`Total_time = Timestamps[-1] - Timestamps[0]`, in seconds.

In [ ]:
CITY = "quebec"
RDA  = "traveltimeCLT/data/trips.rda"   # git clone https://github.com/melmasri/traveltimeCLT
OUT  = "."

MIN_LINKS   = 3        # drop degenerate trips
MIN_SECONDS = 30
MAX_SECONDS = 4 * 3600

In [ ]:
# pip install pandas pyreadr
import ast, json
import numpy as np
import pandas as pd
import pyreadr

In [ ]:
# pyreadr reads the .rda directly; if it is unavailable, export from R with
#   library(traveltimeCLT); data(trips); write.csv(trips, "trips.csv", row.names = FALSE)
trips = pyreadr.read_r(RDA)["trips"]
trips["tripID"] = trips["tripID"].astype("int64")
trips["linkID"] = trips["linkID"].astype("int64")
print(trips.shape)
trips.head()

In [ ]:
print("trips  ", trips.tripID.nunique())
print("links  ", trips.linkID.nunique())
print("period ", trips.entry_time.min(), "->", trips.entry_time.max())
print(trips.timeBin.value_counts().to_string())
trips[["speed", "duration_secs", "distance_meters"]].describe()

In [ ]:
# entry_time is the first GPS fix on the link, duration_secs the traversal time,
# so link k+1 starts (up to rounding) where link k ends.
trips = trips.sort_values(["tripID", "entry_time"], kind="mergesort")
trips["entry_epoch"] = ((trips["entry_time"] - pd.Timestamp("1970-01-01"))
                        // pd.Timedelta("1s")).astype("int64")

per_trip = trips.groupby("tripID").agg(
    n_links=("linkID", "size"),
    first=("entry_epoch", "first"),
    last=("entry_epoch", "last"),
    last_dur=("duration_secs", "last"),
    sum_dur=("duration_secs", "sum"),
    meters=("distance_meters", "sum"),
)
per_trip["span"] = per_trip["last"] + per_trip["last_dur"] - per_trip["first"]
# the two clocks agree, which is the sanity check that the links really are contiguous
print("corr(span, sum of link durations) =",
      round(np.corrcoef(per_trip["span"], per_trip["sum_dur"])[0, 1], 4))
per_trip[["n_links", "span", "meters"]].describe()

In [ ]:
keep = per_trip[(per_trip.n_links >= MIN_LINKS)
                & (per_trip.span >= MIN_SECONDS)
                & (per_trip.span <= MAX_SECONDS)].index
print(f"keeping {len(keep)} / {len(per_trip)} trips")
sel = trips[trips.tripID.isin(keep)]

In [ ]:
rows = []
for trip_id, g in sel.groupby("tripID", sort=True):
    osmids = [str(x) for x in g["linkID"]]
    ts = g["entry_epoch"].tolist()
    # the exit of the last link closes the trip
    ts = ts + [int(round(g["entry_epoch"].iloc[-1] + g["duration_secs"].iloc[-1]))]
    osmids = osmids + [osmids[-1]]
    coords = [(None, None)] * len(osmids)          # no geometry is published
    rows.append({"Id": int(trip_id),
                 "Coordinates": str(coords),
                 "OSMids": str(osmids),
                 "Timestamps": str(ts),
                 "Total_time": ts[-1] - ts[0]})

matched = pd.DataFrame(rows, columns=["Id", "Coordinates", "OSMids", "Timestamps", "Total_time"])
matched.to_csv(f"{OUT}/matched_trips_{CITY}.csv", index=True)
print(matched.shape, "median Total_time", matched.Total_time.median(), "s")
matched.head(2)

In [ ]:
# No road network is published, so the only adjacency we know is the one the
# trips walk over.  This is what the Omsk generator does as well.
pairs = set()
for g in sel.groupby("tripID", sort=False)["linkID"]:
    seq = g[1].tolist()
    pairs.update((str(a), str(b)) for a, b in zip(seq, seq[1:]) if a != b)

edge_list = pd.DataFrame(sorted(pairs), columns=["osmid_u", "osmid_v"])
edge_list.to_csv(f"{OUT}/edge_list_directed_{CITY}.csv", index=False)
print(edge_list.shape)
edge_list.head()

In [ ]:
# Per-link statistics are worth keeping: they are the only "network" this dataset
# has, and free features for a baseline (mean speed, typical length, time bin).
link_stats = (sel.groupby("linkID")
                 .agg(n_obs=("speed", "size"),
                      mean_speed_ms=("speed", "mean"),
                      std_speed_ms=("speed", "std"),
                      median_length_m=("distance_meters", "median"),
                      mean_duration_s=("duration_secs", "mean"))
                 .reset_index()
                 .rename(columns={"linkID": "osmid"}))
link_stats["osmid"] = link_stats["osmid"].astype(str)
link_stats.to_csv(f"{OUT}/link_stats_{CITY}.csv", index=False)

# timeBin is a genuine covariate of the original paper — keep it per trip
time_bin = sel.groupby("tripID")["timeBin"].agg(lambda s: s.mode().iloc[0])
time_bin.rename("timeBin").to_csv(f"{OUT}/trip_timebin_{CITY}.csv")
print(link_stats.shape, time_bin.shape)

In [ ]:
def validate_gold(trips_path, edge_list_path=None, geojson_path=None, require_coords=True):
    """Check the produced files against the Harbin/Omsk contract."""
    df = pd.read_csv(trips_path)
    problems = []

    expected = ["Unnamed: 0", "Id", "Coordinates", "OSMids", "Timestamps", "Total_time"]
    if list(df.columns) != expected:
        problems.append(f"columns are {list(df.columns)}, expected {expected}")

    bad_len = bad_eval = bad_total = bad_coord = 0
    osmids_seen = set()
    for _, r in df.iterrows():
        try:
            c = ast.literal_eval(r["Coordinates"])
            o = ast.literal_eval(r["OSMids"])
            t = ast.literal_eval(r["Timestamps"])
        except Exception:
            bad_eval += 1
            continue
        if not (len(c) == len(o) == len(t)):
            bad_len += 1
        if t[-1] - t[0] != r["Total_time"]:
            bad_total += 1
        if require_coords and not all(isinstance(p, tuple) and len(p) == 2 for p in c):
            bad_coord += 1
        osmids_seen.update(map(str, o))

    for label, n in [("rows that do not literal_eval", bad_eval),
                     ("rows with unequal list lengths", bad_len),
                     ("rows where Total_time != Timestamps[-1] - Timestamps[0]", bad_total),
                     ("rows with malformed coordinates", bad_coord)]:
        if n:
            problems.append(f"{n} {label}")

    print(f"{trips_path}: {len(df)} trips, {len(osmids_seen)} distinct segments, "
          f"Total_time median {df['Total_time'].median():.0f} s")

    if edge_list_path:
        el = pd.read_csv(edge_list_path, dtype=str)
        if list(el.columns) != ["osmid_u", "osmid_v"]:
            problems.append(f"edge list columns are {list(el.columns)}")
        known = set(el["osmid_u"]) | set(el["osmid_v"])
        missing = osmids_seen - known
        print(f"{edge_list_path}: {len(el)} transitions, "
              f"{len(osmids_seen & known)}/{len(osmids_seen)} trip segments present")
        if missing and len(missing) > 0.05 * max(len(osmids_seen), 1):
            problems.append(f"{len(missing)} trip segments missing from the edge list")

    if geojson_path:
        with open(geojson_path) as f:
            gj = json.load(f)
        keys = {f["properties"].get("unique_osmid", f["properties"]["osmid"])
                for f in gj["features"]}
        print(f"{geojson_path}: {len(gj['features'])} features, {len(keys)} unique ids")
        if not osmids_seen <= keys:
            problems.append(f"{len(osmids_seen - keys)} trip segments missing from the geojson")

    print("\nOK — matches the gold contract" if not problems
          else "\nPROBLEMS:\n  " + "\n  ".join(problems))
    return df

In [ ]:
_ = validate_gold(f"{OUT}/matched_trips_{CITY}.csv",
                  f"{OUT}/edge_list_directed_{CITY}.csv",
                  geojson_path=None,
                  require_coords=False)   # Quebec has no coordinates

## Caveats

* **No geometry, and no way to get it.** `linkID` is an internal id of the
  Brisk Synergies network; the mapping to OSM was never published and the exact
  collection period is confidential. Anything in your pipeline that needs
  coordinates (map images, spatial embeddings, GNN features from node positions)
  cannot run on Quebec. What *does* run: sequence-of-segments models, per-segment
  speed baselines, and the whole conformal / calibration line of work.
* `Coordinates` is a list of `(None, None)` of the right length so the file still
  parses with `ast.literal_eval` and passes the length check. Do not feed it to a
  geometry consumer without checking.
* **One fix per link, not per GPS ping.** The published table is already
  aggregated to link level, so the trip has as many "points" as it has links —
  denser than Omsk (multiple fixes per edge), coarser than raw GPS.
* Only 114 `Weekendday` and 112 `EveningNight` rows out of 322 799: the sample is
  overwhelmingly weekday rush hour. Do not use it to study time-of-day transfer.
* The paper (Elmasri et al., *Annals of Applied Statistics*) works with 19 967
  trips; 4 914 were released. If you want the full set, write to the author.
* Licence: the **package** is GPL-3, the data carries no separate statement.
  Ship the loader, not a repackaged copy, until that is clarified.